In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/e-commerce_recommendation/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/e-commerce_recommendation/aisles.csv,aisles.csv,2603,1782368522000
dbfs:/Volumes/workspace/default/e-commerce_recommendation/departments.csv,departments.csv,270,1782368522000
dbfs:/Volumes/workspace/default/e-commerce_recommendation/order_products__prior.csv,order_products__prior.csv,577550706,1782368557000
dbfs:/Volumes/workspace/default/e-commerce_recommendation/order_products__train.csv,order_products__train.csv,24680147,1782368528000
dbfs:/Volumes/workspace/default/e-commerce_recommendation/orders.csv,orders.csv,108968645,1782368557000
dbfs:/Volumes/workspace/default/e-commerce_recommendation/products.csv,products.csv,2166953,1782368523000


In [0]:
BASE_PATH = "/Volumes/workspace/default/e-commerce_recommendation/"

## Orders.CsV

In [0]:
orders_df = spark.read.csv(
    "/Volumes/workspace/default/e-commerce_recommendation/orders.csv",
    header=True,
    inferSchema=True
)

display(orders_df.limit(5))

order_id,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order
2539329,1,prior,1,2,8,null
2398795,1,prior,2,3,7,15.0
473747,1,prior,3,3,12,21.0
2254736,1,prior,4,4,7,29.0
431534,1,prior,5,4,15,28.0


In [0]:
orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- eval_set: string (nullable = true)
 |-- order_number: integer (nullable = true)
 |-- order_dow: integer (nullable = true)
 |-- order_hour_of_day: integer (nullable = true)
 |-- days_since_prior_order: double (nullable = true)



In [0]:
orders_df.count()

3421083

In [0]:
len(orders_df.columns)

7

In [0]:
orders_df.select("user_id").distinct().count()

206209

In [0]:
from pyspark.sql.functions import col, sum

orders_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in orders_df.columns
    ]
).show()

+--------+-------+--------+------------+---------+-----------------+----------------------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|
+--------+-------+--------+------------+---------+-----------------+----------------------+
|       0|      0|       0|           0|        0|                0|                206209|
+--------+-------+--------+------------+---------+-----------------+----------------------+



In [0]:
orders_df.groupBy("eval_set").count().show()

+--------+-------+
|eval_set|  count|
+--------+-------+
|   train| 131209|
|   prior|3214874|
|    test|  75000|
+--------+-------+



In [0]:
orders_df.groupBy("user_id").count().agg({"count":"avg"}).show()

+------------------+
|        avg(count)|
+------------------+
|16.590367054784224|
+------------------+



In [0]:
orders_df.groupBy("order_hour_of_day").count().orderBy("count", ascending=False).show(10)


+-----------------+------+
|order_hour_of_day| count|
+-----------------+------+
|               10|288418|
|               11|284728|
|               15|283639|
|               14|283042|
|               13|277999|
|               12|272841|
|               16|272553|
|                9|257812|
|               17|228795|
|               18|182912|
+-----------------+------+
only showing top 10 rows


In [0]:
orders_df.groupBy("order_dow").count().orderBy("count", ascending=False).show()

+---------+------+
|order_dow| count|
+---------+------+
|        0|600905|
|        1|587478|
|        2|467260|
|        5|453368|
|        6|448761|
|        3|436972|
|        4|426339|
+---------+------+



products.csv - DATASET 


In [0]:
products_df = spark.read.csv(
    BASE_PATH + "products.csv",
    header=True,
    inferSchema=True
)

display(products_df.limit(5))

product_id,product_name,aisle_id,department_id
1,Chocolate Sandwich Cookies,61,19
2,All-Seasons Salt,104,13
3,Robust Golden Unsweetened Oolong Tea,94,7
4,Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce,38,1
5,Green Chile Anytime Sauce,5,13


In [0]:
products_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- aisle_id: string (nullable = true)
 |-- department_id: string (nullable = true)



In [0]:
products_df.count()

49688

In [0]:
products_df.select("department_id").distinct().count()

22

In [0]:
from pyspark.sql.functions import col, sum

products_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in products_df.columns
    ]
).show()

+----------+------------+--------+-------------+
|product_id|product_name|aisle_id|department_id|
+----------+------------+--------+-------------+
|         0|           0|       0|            0|
+----------+------------+--------+-------------+



In [0]:
products_df.show(5, truncate=False)

+----------+-----------------------------------------------------------------+--------+-------------+
|product_id|product_name                                                     |aisle_id|department_id|
+----------+-----------------------------------------------------------------+--------+-------------+
|1         |Chocolate Sandwich Cookies                                       |61      |19           |
|2         |All-Seasons Salt                                                 |104     |13           |
|3         |Robust Golden Unsweetened Oolong Tea                             |94      |7            |
|4         |Smart Ones Classic Favorites Mini Rigatoni With Vodka Cream Sauce|38      |1            |
|5         |Green Chile Anytime Sauce                                        |5       |13           |
+----------+-----------------------------------------------------------------+--------+-------------+
only showing top 5 rows


## **aisles.csv**

In [0]:
aisles_df = spark.read.csv(
    BASE_PATH + "aisles.csv",
    header=True,
    inferSchema=True
)

display(aisles_df)

aisle_id,aisle
1,prepared soups salads
2,specialty cheeses
3,energy granola bars
4,instant foods
5,marinades meat preparation
6,other
7,packaged meat
8,bakery desserts
9,pasta sauce
10,kitchen supplies


In [0]:
aisles_df.count()

134

## departments.csv

In [0]:
departments_df = spark.read.csv(
    BASE_PATH + "departments.csv",
    header=True,
    inferSchema=True
)

display(departments_df)

department_id,department
1,frozen
2,other
3,bakery
4,produce
5,alcohol
6,international
7,beverages
8,pets
9,dry goods pasta
10,bulk


In [0]:
departments_df.printSchema()

root
 |-- department_id: integer (nullable = true)
 |-- department: string (nullable = true)



In [0]:
departments_df.count()

21

### order_products__prior.csv

In [0]:
prior_df = spark.read.csv(
    BASE_PATH + "order_products__prior.csv",
    header=True,
    inferSchema=True
)

display(prior_df.limit(5))

order_id,product_id,add_to_cart_order,reordered
2,33120,1,1
2,28985,2,1
2,9327,3,0
2,45918,4,1
2,30035,5,0


In [0]:
prior_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



In [0]:
prior_df.count()

32434489

In [0]:
prior_df.select("reordered").distinct().show()

+---------+
|reordered|
+---------+
|        1|
|        0|
+---------+



In [0]:
prior_df.select("product_id").distinct().count()

49677

In [0]:
prior_df.select("order_id").distinct().count()

3214874

### Order_Products_Train.CSV

In [0]:
train_df = spark.read.csv(
    BASE_PATH + "order_products__train.csv",
    header=True,
    inferSchema=True
)

display(train_df.limit(5))

order_id,product_id,add_to_cart_order,reordered
1,49302,1,1
1,11109,2,1
1,10246,3,0
1,49683,4,0
1,43633,5,1


In [0]:
train_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- add_to_cart_order: integer (nullable = true)
 |-- reordered: integer (nullable = true)



In [0]:
train_df.count()

1384617

In [0]:
train_df.select("order_id").distinct().count()

131209

In [0]:
train_df.select("product_id").distinct().count()

39123

In [0]:
train_df.groupBy("reordered").count().show()

+---------+------+
|reordered| count|
+---------+------+
|        1|828824|
|        0|555793|
+---------+------+



In [0]:
from pyspark.sql.functions import col, sum

train_df.select(
    [
        sum(col(c).isNull().cast("int")).alias(c)
        for c in train_df.columns
    ]
).show()

+--------+----------+-----------------+---------+
|order_id|product_id|add_to_cart_order|reordered|
+--------+----------+-----------------+---------+
|       0|         0|                0|        0|
+--------+----------+-----------------+---------+



In [0]:
display(train_df.limit(10))

order_id,product_id,add_to_cart_order,reordered
1,49302,1,1
1,11109,2,1
1,10246,3,0
1,49683,4,0
1,43633,5,1
1,13176,6,0
1,47209,7,0
1,22035,8,1
36,39612,1,0
36,19660,2,1
